# **Libs Install**

In [2]:
import sys
!{sys.executable} -m pip install pycryptodomex

# **Question - 1**

In [3]:
import hmac
from Cryptodome.Cipher import AES

def HexToBytes(HexStr):
    return bytes.fromhex(HexStr)

def BytesToHex(B):
    return B.hex()

def XorBytes(A, B):
    return bytes(x ^ y for x, y in zip(A, B))

def Inc32(CounterBlock16):
    Prefix = CounterBlock16[:12]
    Ctr = int.from_bytes(CounterBlock16[12:], "big")
    Ctr = (Ctr + 1) & 0xFFFFFFFF
    return Prefix + Ctr.to_bytes(4, "big")

RConst = 0xE1000000000000000000000000000000

def MulGf128(X, Y):
    Z = 0
    V = Y
    for i in range(128):
        if (X >> (127 - i)) & 1:
            Z ^= V
        if (V & 1) == 0:
            V >>= 1
        else:
            V = (V >> 1) ^ RConst
    return Z

def PadTo16(Data):
    r = len(Data) % 16
    if r == 0:
        return Data
    return Data + b"\x00" * (16 - r)

def Ghash(H, Aad, Ct):
    HInt = int.from_bytes(H, "big")
    Y = 0

    AadP = PadTo16(Aad)
    for off in range(0, len(AadP), 16):
        blk = int.from_bytes(AadP[off:off + 16], "big")
        Y = MulGf128(Y ^ blk, HInt)

    CtP = PadTo16(Ct)
    for off in range(0, len(CtP), 16):
        blk = int.from_bytes(CtP[off:off + 16], "big")
        Y = MulGf128(Y ^ blk, HInt)

    LenBlk = (len(Aad) * 8).to_bytes(8, "big") + (len(Ct) * 8).to_bytes(8, "big")
    Y = MulGf128(Y ^ int.from_bytes(LenBlk, "big"), HInt)

    return Y.to_bytes(16, "big")

def AesBlockEnc(Key, Block16):
    return AES.new(Key, AES.MODE_ECB).encrypt(Block16)

def CtrCrypt(Key, StartCounter16, Data):
    Ecb = AES.new(Key, AES.MODE_ECB)
    Out = bytearray()
    Ctr = StartCounter16

    for off in range(0, len(Data), 16):
        chunk = Data[off:off + 16]
        ks = Ecb.encrypt(Ctr)
        Out.extend(bytes(b ^ k for b, k in zip(chunk, ks[:len(chunk)])))
        Ctr = Inc32(Ctr)

    return bytes(Out)

def GcmEncrypt(Key, Nonce12, Pt, Aad=b""):
    if len(Key) != 16:
        raise ValueError("Key must be 16 bytes (AES-128).")
    if len(Nonce12) != 12:
        raise ValueError("Nonce must be 12 bytes (96-bit).")

    H = AesBlockEnc(Key, b"\x00" * 16)
    J0 = Nonce12 + b"\x00\x00\x00\x01"

    Ct = CtrCrypt(Key, Inc32(J0), Pt)
    S = Ghash(H, Aad, Ct)
    Tag = XorBytes(AesBlockEnc(Key, J0), S)

    return Ct, Tag

def GcmDecryptVerify(Key, Nonce12, Ct, Tag, Aad=b""):
    if len(Key) != 16:
        raise ValueError("Key must be 16 bytes (AES-128).")
    if len(Nonce12) != 12:
        raise ValueError("Nonce must be 12 bytes (96-bit).")

    H = AesBlockEnc(Key, b"\x00" * 16)
    J0 = Nonce12 + b"\x00\x00\x00\x01"

    S = Ghash(H, Aad, Ct)
    TagExpected = XorBytes(AesBlockEnc(Key, J0), S)
    Ok = hmac.compare_digest(TagExpected, Tag)

    Pt = CtrCrypt(Key, Inc32(J0), Ct)

    return Pt, Ok, TagExpected

pt_hex = "00112233445566778899aabbccddeeff"
key_hex = "000102030405060708090a0b0c0d0e0f"
nonce_hex = "aabbccddeeff001122334455"
aad_hex = ""

Pt = HexToBytes(pt_hex)
Key = HexToBytes(key_hex)
Nonce = HexToBytes(nonce_hex)
Aad = HexToBytes(aad_hex) if aad_hex else b""

Ct, Tag = GcmEncrypt(Key, Nonce, Pt, Aad)
Pt2, Ok, TagExpected = GcmDecryptVerify(Key, Nonce, Ct, Tag, Aad)

print("=== AES-128 GCM ===")
print(f"Key (K)           : {BytesToHex(Key)}")
print(f"96-bit Nonce (N)  : {BytesToHex(Nonce)}")
print(f"Plaintext (P)     : {BytesToHex(Pt)}")
print(f"Ciphertext (C)    : {BytesToHex(Ct)}")
print(f"Auth Tag (T)      : {BytesToHex(Tag)}")
print()
print("=== Decryption & Tag Verification ===")
print(f"Decrypted Plaintext: {BytesToHex(Pt2)}")
print(f"Recomputed Tag     : {BytesToHex(TagExpected)}")
print(f"Tag Verified?      : {Ok}")

=== AES-128 GCM ===
Key (K)           : 000102030405060708090a0b0c0d0e0f
96-bit Nonce (N)  : aabbccddeeff001122334455
Plaintext (P)     : 00112233445566778899aabbccddeeff
Ciphertext (C)    : 62db4e89c0ca387a1933c394808e9158
Auth Tag (T)      : b4a00ef1da31110bbe4448ddc89c8b92

=== Decryption & Tag Verification ===
Decrypted Plaintext: 00112233445566778899aabbccddeeff
Recomputed Tag     : b4a00ef1da31110bbe4448ddc89c8b92
Tag Verified?      : True


In [4]:
GfIdentity = 1 << 127
def PowGf128(A, E):
    R = GfIdentity
    B = A
    while E > 0:
        if E & 1:
            R = MulGf128(R, B)
        B = MulGf128(B, B)
        E >>= 1
    return R

def InvGf128(A):
    if A == 0:
        raise ValueError("No inverse for 0 in GF(2^128).")
    return PowGf128(A, (1 << 128) - 2)

def SqrtGf128(A):
    return PowGf128(A, 1 << 127)

Key = HexToBytes("000102030405060708090a0b0c0d0e0f")
Nonce = HexToBytes("aabbccddeeff001122334455")
Aad = b""

P1 = HexToBytes("00112233445566778899aabbccddeeff")
P2 = HexToBytes("102132435465768798a9babbdcddedef")

C1, T1 = GcmEncrypt(Key, Nonce, P1, Aad)
C2, T2 = GcmEncrypt(Key, Nonce, P2, Aad)

print("=== Nonce Reuse Attack Demo (Same Key + Same Nonce) ===")
print(f"K: {BytesToHex(Key)}")
print(f"N: {BytesToHex(Nonce)}")
print()
print("Message 1")
print(f"P1: {BytesToHex(P1)}")
print(f"C1: {BytesToHex(C1)}")
print(f"T1: {BytesToHex(T1)}")
print()
print("Message 2")
print(f"P2: {BytesToHex(P2)}")
print(f"C2: {BytesToHex(C2)}")
print(f"T2: {BytesToHex(T2)}")
print()

XorC = XorBytes(C1, C2)
XorP = XorBytes(P1, P2)

print("=== Confidentiality Break (CTR Keystream Reuse) ===")
print("C1 = P1 XOR KS,  C2 = P2 XOR KS")
print("So: C1 XOR C2 = P1 XOR P2")
print(f"C1 XOR C2: {BytesToHex(XorC)}")
print(f"P1 XOR P2: {BytesToHex(XorP)}")
print(f"Equal?    : {XorC == XorP}")
print()

RecoveredP2 = XorBytes(XorC, P1)
print("If attacker knows P1, they recover P2:")
print("P2 = (C1 XOR C2) XOR P1")
print(f"Recovered P2: {BytesToHex(RecoveredP2)}")
print(f"Matches P2? : {RecoveredP2 == P2}")
print()

Keystream = XorBytes(C1, P1)
print("Recover keystream if one plaintext is known:")
print("KS = C1 XOR P1")
print(f"Recovered KS: {BytesToHex(Keystream)}")
print()

print("=== Authenticity Break (Tag Relation + Forgery) ===")
print("T = E(K,J0) XOR GHASH_H(A, C)")
print("Same nonce => same J0 => same E(K,J0)")
print("So: T1 XOR T2 = GHASH_H(A,C1) XOR GHASH_H(A,C2)")
print()

HReal = AesBlockEnc(Key, b"\x00" * 16)
DeltaT = int.from_bytes(XorBytes(T1, T2), "big")
DeltaC = int.from_bytes(XorBytes(C1, C2), "big")

H2 = MulGf128(DeltaT, InvGf128(DeltaC))
HAtk = SqrtGf128(H2)
HAtkBytes = HAtk.to_bytes(16, "big")

EJ0 = XorBytes(T1, Ghash(HAtkBytes, Aad, C1))

print("1-block, empty AAD case:")
print("T1 XOR T2 = (C1 XOR C2) * H^2")
print("So attacker gets H^2, then H = sqrt(H^2), then can compute E(K,J0)")
print(f"Recovered H (attacker): {BytesToHex(HAtkBytes)}")
print(f"Real H (from key)     : {BytesToHex(HReal)}")
print(f"H correct?            : {HAtkBytes == HReal}")
print()

P3 = HexToBytes("ffffffffffffffff0000000000000000")
C3 = XorBytes(P3, Keystream)
T3 = XorBytes(EJ0, Ghash(HAtkBytes, Aad, C3))

P3Out, Ok3, _ = GcmDecryptVerify(Key, Nonce, C3, T3, Aad)

print("Forge a new valid (C3, T3) under reused nonce:")
print("Choose P3, set C3 = P3 XOR KS, set T3 = E(K,J0) XOR GHASH_H(A,C3)")
print(f"P3 (chosen) : {BytesToHex(P3)}")
print(f"C3 (forged) : {BytesToHex(C3)}")
print(f"T3 (forged) : {BytesToHex(T3)}")
print(f"Tag Verified? : {Ok3}")
print(f"Decrypt Out   : {BytesToHex(P3Out)}")
print(f"Matches P3?   : {P3Out == P3}")

=== Nonce Reuse Attack Demo (Same Key + Same Nonce) ===
K: 000102030405060708090a0b0c0d0e0f
N: aabbccddeeff001122334455

Message 1
P1: 00112233445566778899aabbccddeeff
C1: 62db4e89c0ca387a1933c394808e9158
T1: b4a00ef1da31110bbe4448ddc89c8b92

Message 2
P2: 102132435465768798a9babbdcddedef
C2: 72eb5ef9d0fa288a0903d394908e9248
T2: 773aae7829ff18c03f77562628665e12

=== Confidentiality Break (CTR Keystream Reuse) ===
C1 = P1 XOR KS,  C2 = P2 XOR KS
So: C1 XOR C2 = P1 XOR P2
C1 XOR C2: 10301070103010f01030100010000310
P1 XOR P2: 10301070103010f01030100010000310
Equal?    : True

If attacker knows P1, they recover P2:
P2 = (C1 XOR C2) XOR P1
Recovered P2: 102132435465768798a9babbdcddedef
Matches P2? : True

Recover keystream if one plaintext is known:
KS = C1 XOR P1
Recovered KS: 62ca6cba849f5e0d91aa692f4c537fa7

=== Authenticity Break (Tag Relation + Forgery) ===
T = E(K,J0) XOR GHASH_H(A, C)
Same nonce => same J0 => same E(K,J0)
So: T1 XOR T2 = GHASH_H(A,C1) XOR GHASH_H(A,C2)

1-block, emp